In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Load unified data cube (fact + all dimensions already joined)
cube = spark.read.table("subscription_pipeline.gold.data_cube")

print("✓ Data cube loaded")
print(f"Total records: {cube.count():,}")
print(f"Total columns: {len(cube.columns)}")

## 1.⁠ ⁠How many customers did we gain and lose this period? 

In [0]:
# KPI 1: How many customers did we gain and lose this period?

# CUSTOMERS GAINED: New customers per year
customers_gained = cube.filter(
    col("is_new_customer") == True
).groupBy("year").agg(
    countDistinct("customer_id").alias("customers_gained")
)

# CUSTOMERS LOST: Customers whose FINAL subscription ended per year
# A customer is only counted as lost when their last subscription ends
window_last_sub = Window.partitionBy("customer_id").orderBy(
    col("end_date").desc_nulls_last(),
    col("start_date").desc()
)

customers_lost = cube.withColumn(
    "is_last_subscription",
    row_number().over(window_last_sub) == 1
).filter(
    (col("is_last_subscription") == True) &
    (col("end_date").isNotNull())
).withColumn(
    "year_lost", 
    year(col("end_date"))
).groupBy("year_lost").agg(
    countDistinct("customer_id").alias("customers_lost")
).withColumnRenamed("year_lost", "year")

# Combine gains and losses
kpi1_result = customers_gained.join(customers_lost, "year", "full_outer") \
    .fillna(0) \
    .withColumn("net_change", col("customers_gained") - col("customers_lost")) \
    .orderBy("year")

print("="*60)
print("KPI 1: Customer Gains and Losses (By Year)")
print("="*60)
display(kpi1_result)

## 2.⁠ ⁠Are we losing more customers now compared to previous periods?


In [0]:
# Reuse the customer loss data from KPI 1
loss_trend = kpi1_result.select(
    col("year"),
    col("customers_lost")
).orderBy("year")

# Calculate period-over-period changes
window_spec = Window.orderBy("year")

loss_comparison = loss_trend.withColumn(
    "prev_period_lost",
    lag("customers_lost").over(window_spec)
).withColumn(
    "change",
    col("customers_lost") - col("prev_period_lost")
).withColumn(
    "change_pct",
    when(col("prev_period_lost") > 0, 
         round((col("customers_lost") - col("prev_period_lost")) / col("prev_period_lost") * 100, 1))
    .otherwise(None)
)

print("="*70)
print("KPI 2: Customer Loss Trend Analysis")
print("="*70)
display(loss_comparison)

# Summary analysis
latest_year_row = loss_comparison.orderBy(col("year").desc()).first()
current_year_val = latest_year_row["year"]
current_losses = latest_year_row["customers_lost"]
prev_losses = latest_year_row["prev_period_lost"]
change = latest_year_row["change"]
change_pct = latest_year_row["change_pct"]

print("\n" + "="*70)
print("SUMMARY")
print("="*70)
print(f"Latest Period ({current_year_val}): {current_losses:,} customers lost")
if prev_losses is not None and prev_losses > 0:
    print(f"Previous Period ({current_year_val-1}): {prev_losses:,} customers lost")
    print(f"Change: {change:+,} ({change_pct:+.1f}%)")
    
    if change > 0:
        print(f"\n⚠️  YES - We are losing MORE customers now.")
        print(f"   Customer losses increased by {change:,} customers ({change_pct:.1f}%)")
    elif change < 0:
        print(f"\n✓ NO - Customer losses have DECREASED.")
        print(f"  Customer losses dropped by {-change:,} customers ({-change_pct:.1f}%)")
    else:
        print(f"\n→ Customer losses remained FLAT.")
else:
    print("No previous period data for comparison")
print("="*70)

## 3.⁠ ⁠Who are our highest-value customers by recurring revenue? 


In [0]:
# KPI 3: Who are our highest-value customers by recurring revenue?

# Calculate total recurring revenue per customer
# All customer details are already in the cube (customer_name, customer_country, industry_type, customer_type)
highest_value_customers = cube.groupBy("customer_id").agg(
    sum("revenue_gbp").alias("total_recurring_revenue"),
    count("*").alias("total_subscriptions"),
    min("start_date").alias("first_subscription_date"),
    max("start_date").alias("latest_subscription_date"),
    sum(when(col("is_active_subscription") == True, 1).otherwise(0)).alias("active_subscriptions"),
    first("customer_name").alias("customer_name"),
    first("customer_country").alias("country"),
    first("industry_type").alias("industry_type"),
    first("customer_type").alias("customer_type")
).orderBy(col("total_recurring_revenue").desc())

print("="*70)
print("KPI 3: Highest-Value Customers by Recurring Revenue")
print("="*70)
print("\nTop 20 Customers:")
display(highest_value_customers.limit(20))

# Summary statistics
top_10_revenue = highest_value_customers.limit(10).agg(sum("total_recurring_revenue")).collect()[0][0]
total_revenue = highest_value_customers.agg(sum("total_recurring_revenue")).collect()[0][0]
total_customers = highest_value_customers.count()

print(f"\n{'='*70}")
print("SUMMARY")
print("="*70)
print(f"Total Customers: {total_customers:,}")
if total_revenue:
    print(f"Total Recurring Revenue: £{total_revenue:,.2f}")
    print(f"Top 10 Customers Revenue: £{top_10_revenue:,.2f} ({top_10_revenue/total_revenue*100:.1f}% of total)")
    print(f"Average Revenue per Customer: £{total_revenue/total_customers:,.2f}")
else:
    print("No revenue data available")
print("="*70)

## 4.⁠ ⁠Which customers are expanding their product usage over time?


In [0]:
# KPI 4: Which customers are expanding their product usage over time?

# Identify customers with upgrades or positive revenue growth
# All flags (is_upgrade, revenue_change) and customer details are already in cube
expanding_customers = cube.groupBy("customer_id").agg(
    sum(when(col("is_upgrade") == True, 1).otherwise(0)).alias("total_upgrades"),
    sum(when(col("revenue_change") > 0, col("revenue_change")).otherwise(0)).alias("total_revenue_increase"),
    sum("revenue_gbp").alias("total_revenue"),
    count("*").alias("total_subscriptions"),
    min("start_date").alias("first_subscription"),
    max("start_date").alias("latest_subscription"),
    first("customer_name").alias("customer_name"),
    first("customer_country").alias("country"),
    first("industry_type").alias("industry_type"),
    first("customer_type").alias("customer_type")
).filter(
    # Customers with at least 1 upgrade OR positive revenue growth
    (col("total_upgrades") > 0) | (col("total_revenue_increase") > 0)
).orderBy(col("total_upgrades").desc(), col("total_revenue_increase").desc())

print("="*70)
print("KPI 4: Customers Expanding Product Usage")
print("="*70)
print("\nTop 20 Expanding Customers (by upgrades):")
display(expanding_customers.limit(20))

# Summary statistics
total_expanding = expanding_customers.count()
total_upgrades = expanding_customers.agg(sum("total_upgrades")).collect()[0][0]
avg_upgrades = expanding_customers.filter(col("total_upgrades") > 0).agg(avg("total_upgrades")).collect()[0][0]
total_revenue_growth = expanding_customers.agg(sum("total_revenue_increase")).collect()[0][0]

print(f"\n{'='*70}")
print("SUMMARY")
print("="*70)
print(f"Customers Expanding Usage: {total_expanding:,}")
print(f"Total Upgrades Across All Customers: {int(total_upgrades):,}")
print(f"Average Upgrades per Expanding Customer: {avg_upgrades:.1f}")
print(f"Total Revenue Increase from Expansion: £{total_revenue_growth:,.2f}")
print("="*70)

#KPI 5: Which customers are reducing or stopping product usage?

In [0]:
# KPI 5: Which customers are reducing or stopping product usage?

# Identify customers with downgrades or negative revenue changes
# All flags (is_downgrade, revenue_change) and customer details are already in cube
reducing_customers = cube.groupBy("customer_id").agg(
    sum(when(col("is_downgrade") == True, 1).otherwise(0)).alias("total_downgrades"),
    sum(when(col("revenue_change") < 0, col("revenue_change")).otherwise(0)).alias("total_revenue_decrease"),
    sum("revenue_gbp").alias("total_revenue"),
    count("*").alias("total_subscriptions"),
    sum(when(col("subscription_status") == "Closed - Lost", 1).otherwise(0)).alias("lost_subscriptions"),
    sum(when(col("is_active_subscription") == True, 1).otherwise(0)).alias("active_subscriptions"),
    min("start_date").alias("first_subscription"),
    max("start_date").alias("latest_subscription"),
    first("customer_name").alias("customer_name"),
    first("customer_country").alias("country"),
    first("industry_type").alias("industry_type"),
    first("customer_type").alias("customer_type")
).filter(
    # Customers with at least 1 downgrade OR negative revenue change
    (col("total_downgrades") > 0) | (col("total_revenue_decrease") < 0)
).orderBy(col("total_downgrades").desc(), col("total_revenue_decrease").asc())

print("="*70)
print("KPI 5: Customers Reducing or Stopping Product Usage")
print("="*70)
print("\nTop 20 At-Risk Customers (by downgrades):")
display(reducing_customers.limit(20))

# Summary statistics
total_reducing = reducing_customers.count()
total_downgrades = reducing_customers.agg(sum("total_downgrades")).collect()[0][0]
avg_downgrades = reducing_customers.filter(col("total_downgrades") > 0).agg(avg("total_downgrades")).collect()[0][0]
total_revenue_loss = reducing_customers.agg(sum("total_revenue_decrease")).collect()[0][0]
total_lost_subs = reducing_customers.agg(sum("lost_subscriptions")).collect()[0][0]

print(f"\n{'='*70}")
print("SUMMARY")
print("="*70)
print(f"Customers Reducing Usage: {total_reducing:,}")
print(f"Total Downgrades Across All Customers: {int(total_downgrades):,}")
print(f"Average Downgrades per Reducing Customer: {avg_downgrades:.1f}")
print(f"Total Revenue Loss from Reduction: £{-total_revenue_loss:,.2f}")
print(f"Total Lost Subscriptions: {int(total_lost_subs):,}")
print("="*70)

## 6.⁠ ⁠How much recurring revenue are we retaining from our existing customers in current FY?

In [0]:
# KPI 6: How much recurring revenue are we retaining from existing customers in current FY?

# Determine current financial year from cube
current_fy = cube.select(max("financial_year")).collect()[0][0]
print(f"Current Financial Year: FY{current_fy}\n")

# Revenue from EXISTING customers (not new) in current FY
revenue_existing_customers = cube.filter(
    (col("financial_year") == current_fy) &
    (col("is_new_customer") == False)
).agg(
    sum("revenue_gbp").alias("retained_revenue")
).collect()[0][0]

# Revenue from NEW customers in current FY
revenue_new_customers = cube.filter(
    (col("financial_year") == current_fy) &
    (col("is_new_customer") == True)
).agg(
    sum("revenue_gbp").alias("new_revenue")
).collect()[0][0]

# Total revenue in current FY
total_revenue_fy = cube.filter(
    col("financial_year") == current_fy
).agg(
    sum("revenue_gbp").alias("total_revenue")
).collect()[0][0]

# Customer count breakdown
existing_customer_count = cube.filter(
    (col("financial_year") == current_fy) &
    (col("is_new_customer") == False)
).select("customer_id").distinct().count()

new_customer_count = cube.filter(
    (col("financial_year") == current_fy) &
    (col("is_new_customer") == True)
).select("customer_id").distinct().count()

print("="*70)
print(f"KPI 6: Revenue Retention (FY{current_fy})")
print("="*70)
print(f"\nRevenue from Existing Customers: £{revenue_existing_customers:,.2f}" if revenue_existing_customers else "Revenue from Existing Customers: £0.00")
print(f"Revenue from New Customers: £{revenue_new_customers:,.2f}" if revenue_new_customers else "Revenue from New Customers: £0.00")
print(f"Total Revenue: £{total_revenue_fy:,.2f}" if total_revenue_fy else "Total Revenue: £0.00")

if total_revenue_fy and total_revenue_fy > 0:
    retention_pct = (revenue_existing_customers / total_revenue_fy * 100) if revenue_existing_customers else 0
    print(f"\nRevenue Retention Rate: {retention_pct:.1f}%")
    print(f"(Existing customers contribute {retention_pct:.1f}% of total revenue)")

print(f"\nCustomer Breakdown:")
print(f"  Existing Customers: {existing_customer_count:,}")
print(f"  New Customers: {new_customer_count:,}")
print("="*70)

## 7.⁠ ⁠How much recurring revenue growth is coming from upgrades versus losses in current FY?


In [0]:
# KPI 7: How much recurring revenue growth is coming from upgrades versus losses in current FY?

# Revenue from UPGRADES (positive revenue changes) in current FY
upgrade_revenue = cube.filter(
    (col("financial_year") == current_fy) &
    (col("is_upgrade") == True)
).agg(
    sum("revenue_change").alias("upgrade_revenue")
).collect()[0][0]

# Revenue from DOWNGRADES (negative revenue changes) in current FY
downgrade_revenue = cube.filter(
    (col("financial_year") == current_fy) &
    (col("is_downgrade") == True)
).agg(
    sum("revenue_change").alias("downgrade_revenue")
).collect()[0][0]

# Count of upgrades vs downgrades
upgrade_count = cube.filter(
    (col("financial_year") == current_fy) &
    (col("is_upgrade") == True)
).count()

downgrade_count = cube.filter(
    (col("financial_year") == current_fy) &
    (col("is_downgrade") == True)
).count()

# Net revenue change from upgrades/downgrades
net_revenue_change = (upgrade_revenue or 0) + (downgrade_revenue or 0)

print("="*70)
print(f"KPI 7: Revenue Growth from Upgrades vs Losses (FY{current_fy})")
print("="*70)

print("\nUPGRADES:")
if upgrade_revenue:
    print(f"  Revenue Impact: £{upgrade_revenue:,.2f}")
    print(f"  Number of Upgrades: {upgrade_count:,}")
    print(f"  Average per Upgrade: £{upgrade_revenue/upgrade_count:,.2f}")
else:
    print(f"  Revenue Impact: £0.00")
    print(f"  Number of Upgrades: {upgrade_count:,}")

print("\nDOWNGRADES/LOSSES:")
if downgrade_revenue:
    print(f"  Revenue Impact: £{downgrade_revenue:,.2f}")
    print(f"  Number of Downgrades: {downgrade_count:,}")
    print(f"  Average per Downgrade: £{downgrade_revenue/downgrade_count:,.2f}")
else:
    print(f"  Revenue Impact: £0.00")
    print(f"  Number of Downgrades: {downgrade_count:,}")

print("\n" + "="*70)
print("NET IMPACT")
print("="*70)
print(f"Net Revenue Change: £{net_revenue_change:,.2f}")

if upgrade_revenue and downgrade_revenue:
    # Calculate absolute values for comparison
    upgrade_abs = upgrade_revenue if upgrade_revenue > 0 else -upgrade_revenue
    downgrade_abs = downgrade_revenue if downgrade_revenue > 0 else -downgrade_revenue
    total_movement = upgrade_abs + downgrade_abs
    
    print(f"\nUpgrades contribute: {upgrade_abs/total_movement*100:.1f}% of total revenue movement")
    print(f"Downgrades contribute: {downgrade_abs/total_movement*100:.1f}% of total revenue movement")
    
    if upgrade_abs > downgrade_abs:
        print(f"\n✓ POSITIVE: Upgrades (£{upgrade_abs:,.2f}) outweigh downgrades (£{downgrade_abs:,.2f})")
    else:
        print(f"\n⚠️  NEGATIVE: Downgrades (£{downgrade_abs:,.2f}) outweigh upgrades (£{upgrade_abs:,.2f})")

print("="*70)

   
## 8.⁠ ⁠How does revenue year-to-date compare with the same period last year?

In [0]:
# KPI 8: How does revenue year-to-date compare with the same period last year?

# Determine the latest year and the latest month in that year
latest_year = cube.select(max("year")).collect()[0][0]
latest_month_in_year = cube.filter(col("year") == latest_year).select(max("month")).collect()[0][0]

print(f"Comparing YTD through {latest_year}-{latest_month_in_year:02d}\n")

# Current year YTD revenue (up to the latest month)
current_ytd = cube.filter(
    (col("year") == latest_year) &
    (col("month") <= latest_month_in_year)
).agg(
    sum("revenue_gbp").alias("ytd_revenue")
).collect()[0][0]

# Previous year YTD revenue (same period)
previous_year = latest_year - 1
previous_ytd = cube.filter(
    (col("year") == previous_year) &
    (col("month") <= latest_month_in_year)
).agg(
    sum("revenue_gbp").alias("ytd_revenue")
).collect()[0][0]

# Calculate change
if current_ytd and previous_ytd:
    revenue_change = current_ytd - previous_ytd
    revenue_change_pct = (revenue_change / previous_ytd * 100)
else:
    revenue_change = 0
    revenue_change_pct = 0

print("="*70)
print(f"KPI 8: YTD Revenue Comparison")
print("="*70)
print(f"\n{latest_year} YTD Revenue (through Month {latest_month_in_year}): ", end="")
if current_ytd:
    print(f"£{current_ytd:,.2f}")
else:
    print("£0.00")

print(f"{previous_year} YTD Revenue (through Month {latest_month_in_year}): ", end="")
if previous_ytd:
    print(f"£{previous_ytd:,.2f}")
else:
    print("£0.00")

print("\n" + "="*70)
print("COMPARISON")
print("="*70)

if current_ytd and previous_ytd:
    print(f"YoY Change: £{revenue_change:+,.2f} ({revenue_change_pct:+.1f}%)")
    
    if revenue_change > 0:
        print(f"\n✓ POSITIVE: Revenue is UP by £{revenue_change:,.2f} ({revenue_change_pct:.1f}%)")
    elif revenue_change < 0:
        print(f"\n⚠️  NEGATIVE: Revenue is DOWN by £{-revenue_change:,.2f} ({-revenue_change_pct:.1f}%)")
    else:
        print(f"\n→ FLAT: Revenue is unchanged")
else:
    print("Insufficient data for comparison")

print("="*70)

   
## 9.⁠ ⁠Which customers or products contribute the most to total recurring revenue?

In [0]:
# KPI 9: Which customers or products contribute the most to total recurring revenue?

# Calculate total revenue for percentage calculations
total_revenue_all = cube.agg(sum("revenue_gbp")).collect()[0][0]

print("="*70)
print("KPI 9: Top Revenue Contributors")
print("="*70)
print(f"\nTotal Recurring Revenue: £{total_revenue_all:,.2f}\n")

# TOP CUSTOMERS by revenue contribution
# All customer details (customer_name, country, industry_type) are already in cube
top_customers = cube.groupBy("customer_id").agg(
    sum("revenue_gbp").alias("customer_revenue"),
    first("customer_name").alias("customer_name"),
    first("customer_country").alias("country"),
    first("industry_type").alias("industry_type")
).withColumn(
    "revenue_pct",
    round((col("customer_revenue") / total_revenue_all * 100), 2)
).orderBy(col("customer_revenue").desc())

print("TOP 10 CUSTOMERS BY REVENUE:")
display(top_customers.limit(10))

# TOP PRODUCTS by revenue contribution
# All product details (product_name, plan_name, billing_cycle) are already in cube
top_products = cube.groupBy("product_id").agg(
    sum("revenue_gbp").alias("product_revenue"),
    count("*").alias("subscriptions"),
    first("product_name").alias("product_name"),
    first("plan_name").alias("plan_name"),
    first("billing_cycle").alias("billing_cycle")
).withColumn(
    "revenue_pct",
    round((col("product_revenue") / total_revenue_all * 100), 2)
).orderBy(col("product_revenue").desc())

print("\nTOP 10 PRODUCTS BY REVENUE:")
display(top_products.limit(10))

# Summary statistics
top_10_customers_revenue = top_customers.limit(10).agg(sum("customer_revenue")).collect()[0][0]
top_10_products_revenue = top_products.limit(10).agg(sum("product_revenue")).collect()[0][0]

print(f"\n{'='*70}")
print("SUMMARY")
print("="*70)
print(f"Top 10 Customers: £{top_10_customers_revenue:,.2f} ({top_10_customers_revenue/total_revenue_all*100:.1f}% of total)")
print(f"Top 10 Products: £{top_10_products_revenue:,.2f} ({top_10_products_revenue/total_revenue_all*100:.1f}% of total)")
print("="*70)

   
## 10.⁠ ⁠What is the rolling 12 month recurring revenue trend?

In [0]:
# KPI 10: What is the rolling 12 month recurring revenue trend?

# Group revenue by year-month using cube
monthly_revenue = cube.groupBy("year", "month").agg(
    sum("revenue_gbp").alias("monthly_revenue")
).withColumn(
    "year_month",
    concat(col("year"), lit("-"), lpad(col("month"), 2, "0"))
).orderBy("year", "month")

# Calculate rolling 12-month sum using window function
window_12m = Window.orderBy("year", "month").rowsBetween(-11, 0)

rolling_12m_revenue = monthly_revenue.withColumn(
    "rolling_12m_revenue",
    sum("monthly_revenue").over(window_12m)
).withColumn(
    "months_in_window",
    count("*").over(window_12m)
).filter(
    col("months_in_window") == 12  # Only show periods with full 12 months of data
)

print("="*70)
print("KPI 10: Rolling 12-Month Recurring Revenue Trend")
print("="*70)
print("\nFull Trend (all periods with 12 months of data):")
display(rolling_12m_revenue.select("year_month", "monthly_revenue", "rolling_12m_revenue"))

# Calculate trend metrics
first_period = rolling_12m_revenue.orderBy("year", "month").first()
latest_period = rolling_12m_revenue.orderBy(col("year").desc(), col("month").desc()).first()

first_12m = first_period["rolling_12m_revenue"]
latest_12m = latest_period["rolling_12m_revenue"]
trend_change = latest_12m - first_12m
trend_change_pct = (trend_change / first_12m * 100)

print(f"\n{'='*70}")
print("TREND ANALYSIS")
print("="*70)
print(f"First 12M Period ({first_period['year_month']}): £{first_12m:,.2f}")
print(f"Latest 12M Period ({latest_period['year_month']}): £{latest_12m:,.2f}")
print(f"\nOverall Change: £{trend_change:+,.2f} ({trend_change_pct:+.1f}%)")

if trend_change > 0:
    print(f"\n✓ GROWING: Rolling 12M revenue increased by £{trend_change:,.2f} ({trend_change_pct:.1f}%)")
elif trend_change < 0:
    print(f"\n⚠️  DECLINING: Rolling 12M revenue decreased by £{-trend_change:,.2f} ({-trend_change_pct:.1f}%)")
else:
    print(f"\n→ FLAT: Rolling 12M revenue unchanged")

print("="*70)